# Preprocesamiento del Retail Sales Dataset

**Grupo 2** — Angel Espin · Carlos Ramirez

Este cuaderno documenta la limpieza y derivacion de atributos
realizada sobre el dataset original de Kaggle.

---

## Proceso:
1. Carga del CSV original
2. Validacion de calidad (nulos, duplicados, consistencia aritmetica)
3. Derivacion de nuevos atributos para el dashboard
4. Exportacion del dataset limpio

In [ ]:
# Imports
import pandas as pd
from pathlib import Path

BASE = Path.cwd()
RAW   = BASE / "retail_sales_dataset.csv"
CLEAN = BASE / "retail_sales_clean.csv"

df = pd.read_csv(RAW)
df["Date"] = pd.to_datetime(df["Date"])
print("Dimensiones:", df.shape)
df.head()

In [ ]:
# Validacion de calidad
print("Nulos por columna:\n", df.isnull().sum(), "\n")
print("Filas duplicadas:", df.duplicated().sum())
print("Transaction ID unicos:", df["Transaction ID"].nunique())
print("Customer ID unicos:", df["Customer ID"].nunique())

# Consistencia: Total Amount == Quantity * Price per Unit
inconsistentes = (df["Quantity"] * df["Price per Unit"] != df["Total Amount"]).sum()
print("Filas con Total != Quantity*Price:", inconsistentes)

# Rango temporal
print("Fecha min:", df["Date"].min().date(), "| Fecha max:", df["Date"].max().date())
print("Transacciones por anio:\n", df["Date"].dt.year.value_counts())

In [ ]:
# Derivacion de atributos nuevos
MESES = {1:"01-Enero",2:"02-Febrero",3:"03-Marzo",4:"04-Abril",
         5:"05-Mayo",6:"06-Junio",7:"07-Julio",8:"08-Agosto",
         9:"09-Septiembre",10:"10-Octubre",11:"11-Noviembre",12:"12-Diciembre"}
DIAS  = {0:"1-Lunes",1:"2-Martes",2:"3-Miercoles",3:"4-Jueves",
         4:"5-Viernes",5:"6-Sabado",6:"7-Domingo"}

df["Anio"]      = df["Date"].dt.year
df["Mes"]       = df["Date"].dt.month
df["NombreMes"] = df["Mes"].map(MESES)
df["Trimestre"] = "T" + df["Date"].dt.quarter.astype(str)
df["DiaSemana"] = df["Date"].dt.dayofweek.map(DIAS)
df["TipoDia"]   = df["Date"].dt.dayofweek.apply(
    lambda d: "Fin de semana" if d >= 5 else "Entre semana"
)
df["GrupoEdad"] = pd.cut(df["Age"], bins=[18,25,35,45,55,65],
                         labels=["18-25","26-35","36-45","46-55","56-65"],
                         include_lowest=True, right=True).astype(str)
df["NivelPrecio"] = df["Price per Unit"].apply(
    lambda p: "Bajo (<=50)" if p <= 50 else "Alto (>=300)"
)

df.to_csv(CLEAN, index=False, encoding="utf-8-sig")
print("CSV limpio guardado en:", CLEAN)
print("Atributos finales (", len(df.columns), "):")
print(list(df.columns))
df.head()

In [ ]:
# Resumen final
print("=" * 50)
print("RESUMEN DEL DATASET LIMPIO")
print("=" * 50)
print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"Ingresos totales: ${df['Total Amount'].sum():,.0f}")
print(f"Ticket promedio: ${df['Total Amount'].mean():,.2f}")
print(f"Unidades vendidas: {df['Quantity'].sum():,}")
print(f"Categorias: {df['Product Category'].nunique()}")
print(f"Rango edad: {df['Age'].min()} - {df['Age'].max()}")
print(f"Rango fechas: {df['Date'].min().date()} a {df['Date'].max().date()}")
print(f"Valores nulos: {df.isnull().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")